# Apprentissage et évaluation

Objectif : identifier l’utilisateur d’une session à partir des caractéristiques construites dans `002_features.ipynb`.

| Étape | Traitement |
| --- | --- |
| Caractéristiques | Reconstruire `X` et `y` depuis le deuxième notebook |
| Validation | Réserver 20 % des sessions, avec une séparation stratifiée |
| Prétraitements | Apprendre l’encodage et la mise à l’échelle sur l’apprentissage uniquement |
| Premier modèle | KNN avec trois voisins |
| Comparaison | KNN et SVM linéaire sur la même validation |
| Évaluation | Accuracy, F1 macro, F1 pondéré et analyse des erreurs |

KNN utilise les caractéristiques des sessions : il constitue ici un premier modèle de référence. Le type de moyenne du F1 utilisé par Kaggle reste à confirmer. Le jeu de test Kaggle n’est pas utilisé.

## Vérification des bibliothèques

Pandas et scikit-learn doivent être disponibles dans le noyau utilisé. Aucune installation n’est lancée dans ce notebook.

In [1]:
import importlib.util

for nom in ["pandas", "sklearn"]:
    disponible = importlib.util.find_spec(nom) is not None
    print(f"{nom} disponible : {disponible}")

pandas disponible : True
sklearn disponible : True


## Imports et configuration

In [2]:
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

PROJECT_PATH = Path(
    r"C:\Users\Rog strix Scar\Desktop\BE_MachineLearning"
)

FEATURES_NOTEBOOK = (
    PROJECT_PATH / "datamanip" / "002_features.ipynb"
)

RANDOM_STATE = 42

print("Notebook de caractéristiques trouvé :", FEATURES_NOTEBOOK.is_file())

Notebook de caractéristiques trouvé : True


## Reconstruction des caractéristiques

Les cellules Python du notebook `002_features.ipynb` sont exécutées pour reconstruire les caractéristiques. Cette lecture ne modifie aucun fichier.

In [3]:
import json

with FEATURES_NOTEBOOK.open(encoding="utf-8") as fichier:
    notebook_features = json.load(fichier)

espace_features = {"__name__": "__main__"}

for numero, cellule in enumerate(notebook_features["cells"]):
    if cellule["cell_type"] == "code":
        source = "".join(cellule["source"])

        exec(
            compile(
                source,
                f"{FEATURES_NOTEBOOK.name} — cellule {numero}",
                "exec",
            ),
            espace_features,
        )

X_sessions = espace_features["X"].tolist()
y_sessions = espace_features["y"].copy()

df_raw = espace_features["df_raw"]
df_train = espace_features["df_train"]

assert len(X_sessions) == len(y_sessions)
assert espace_features["X"].index.equals(y_sessions.index)

print("Nombre de sessions :", len(X_sessions))
print("Nombre d’utilisateurs :", y_sessions.nunique())

Fichier trouvé : True
Nombre de sessions : 3279
Nombre d’utilisateurs : 247


,session_id,util,navigateur,traces
0,1,nuh,Firefox,[Création d'un écran(infologic.core.accueil.Ac...
1,2,muz,Google Chrome,[Création d'un écran(infologic.core.gui.contro...
2,3,zrx,Microsoft Edge,[Affichage d'une dialogue(infologic.core.gui.c...
3,4,pou,Firefox,[Création d'un écran(infologic.core.gui.contro...
4,5,ald,Google Chrome,[Affichage d'une dialogue(infologic.acti.modul...


,util,navigateur,types_actions,ecrans_contextes,configurations,chaines
0,nuh,Firefox,"[Création d'un écran, Affichage d'une dialogue...","[infologic.core.accueil.AccueilController, inf...","[DEF_03/24, ACCUEIL_GMAO, 001, ACCUEIL_GMAO, D...","[JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, ..."
1,muz,Google Chrome,"[Création d'un écran, Création d'un écran, Sél...",[infologic.core.gui.controllers.BlankControlle...,"[TELEM_MINIMAL, DEF_03/24, TELEM_MINIMAL, DEF_...","[TEC, TEC, DB, DB, DB, DB, DB, DB, DB, DB, DB,..."
2,zrx,Microsoft Edge,"[Affichage d'une dialogue, Exécution d'un bout...",[infologic.core.gui.controllers.BlankControlle...,"[ACCUEIL_INST, INSTALLATEUR, DEF_07/23, ACCUEI...","[GP, GP, GP, GP, GP, MES, MES, MES, MES, MES, ..."
3,pou,Firefox,"[Création d'un écran, Exécution d'un bouton, A...",[infologic.core.gui.controllers.nested.InputFo...,"[STD, DEF_03/24, STD, DEF_03/24, STD, DEF_03/2...","[JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, ..."
4,ald,Google Chrome,"[Affichage d'une dialogue, Exécution d'un bout...",[infologic.acti.modules.AT_ACTIVITES.compteren...,"[INSTALLATEUR, DEF_03/24, TELEM_MINIMAL, DEF_0...","[ST, ST, ST, GP, GP, GP, GP, GP, GP, GP, GP]"


,util,nb_actions,nb_marqueurs_temporels,dernier_repere_s,sans_actions,sans_marqueur_temporel,nb_actions_modification,part_actions_modification,nb_types_actions_distincts,nb_ecrans_contextes_distincts,nb_configurations_distinctes,nb_chaines_distinctes
0,nuh,2514,614,2905,0,0,4,0.001591,19,7,3,2
1,muz,90,46,230,0,0,26,0.288889,11,5,2,2
2,zrx,608,158,750,0,0,50,0.082237,21,17,7,2
3,pou,886,313,1445,0,0,64,0.072235,19,10,4,1
4,ald,173,60,275,0,0,2,0.011561,14,8,4,2


Session : 1
Utilisateur : nuh


...

Nombre de sessions dans X : 3279
Nombre de réponses dans y : 3279
Nombre de classes : 247


,util,nb_caracteristiques_presentes
0,nuh,62
1,muz,43
2,zrx,80
3,pou,65
4,ald,54


Utilisateur réel : nuh


,valeur
nb_actions,2514
nb_marqueurs_temporels,614
dernier_repere_s,2905
sans_actions,0
sans_marqueur_temporel,0
nb_actions_modification,4
part_actions_modification,0.001591
nb_types_actions_distincts,19
nb_ecrans_contextes_distincts,7
nb_configurations_distinctes,3


Tous les contrôles sont validés.
Nombre de sessions : 3279
Nombre d’utilisateurs : 247


## Séparation entre apprentissage et validation

| Paramètre | Choix |
| --- | --- |
| Part de validation | 20 % |
| Répartition des utilisateurs | Stratifiée |
| Reproductibilité | Graine 42 |

Les deux ensembles doivent contenir les 247 utilisateurs et aucune ligne commune.

In [4]:
X_apprentissage, X_validation, y_apprentissage, y_validation = (
    train_test_split(
        X_sessions,
        y_sessions,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y_sessions,
    )
)

assert set(y_apprentissage) == set(y_sessions)
assert set(y_validation) == set(y_sessions)
assert set(y_apprentissage.index).isdisjoint(y_validation.index)

resume_separation = pd.DataFrame({
    "ensemble": ["Apprentissage", "Validation"],
    "nb_sessions": [
        len(X_apprentissage),
        len(X_validation),
    ],
    "nb_utilisateurs": [
        y_apprentissage.nunique(),
        y_validation.nunique(),
    ],
})

display(resume_separation)

,ensemble,nb_sessions,nb_utilisateurs
0,Apprentissage,2623,247
1,Validation,656,247


## Métriques d’évaluation

| Métrique | Signification |
| --- | --- |
| Accuracy | Proportion de sessions correctement classées |
| F1 macro | Même poids pour chaque utilisateur |
| F1 pondéré | Poids proportionnel au nombre de sessions de validation |

Le type de moyenne utilisé par Kaggle reste à confirmer.

In [5]:
def calculer_scores(y_reel, y_predit):
    return {
        "accuracy": accuracy_score(y_reel, y_predit),
        "f1_macro": f1_score(
            y_reel,
            y_predit,
            average="macro",
            zero_division=0,
        ),
        "f1_pondere": f1_score(
            y_reel,
            y_predit,
            average="weighted",
            zero_division=0,
        ),
    }


resultats = []
modeles = {}
predictions_modeles = {}

## Premier modèle : KNN

| Composant | Choix et rôle |
| --- | --- |
| Encodage | `DictVectorizer` transforme les dictionnaires et encode le navigateur |
| Mise à l’échelle | `StandardScaler(with_mean=False)` conserve la représentation creuse |
| Voisins | Trois sessions proches votent avec le même poids |
| Distance | Euclidienne, calculée sur les caractéristiques mises à l’échelle |
| Recherche | Comparaison directe avec les sessions d’apprentissage |

L’encodage et la mise à l’échelle sont ajustés sur l’apprentissage uniquement. Le navigateur est inclus dans les caractéristiques. Aucun résultat de classification n’est annoncé avant l’exécution de cette cellule.

In [6]:
reference = Pipeline([
    (
        "vectorisation",
        DictVectorizer(sparse=True),
    ),
    (
        "mise_echelle",
        StandardScaler(with_mean=False),
    ),
    (
        "classification",
        KNeighborsClassifier(
            n_neighbors=3,
            weights="uniform",
            metric="euclidean",
            algorithm="brute",
            n_jobs=-1,
        ),
    ),
])

debut = perf_counter()

reference.fit(
    X_apprentissage,
    y_apprentissage,
)

predictions_reference = reference.predict(X_validation)

scores_reference = calculer_scores(
    y_validation,
    predictions_reference,
)

nom_modele = "KNN — 3 voisins"
modeles[nom_modele] = reference
predictions_modeles[nom_modele] = predictions_reference
resultats = [
    ligne for ligne in resultats
    if ligne["modele"] != nom_modele
]

resultats.append({
    "modele": nom_modele,
    **scores_reference,
})

print(
    "Durée d’apprentissage et de prédiction :",
    round(perf_counter() - debut, 2),
    "secondes",
)

print(
    "Nombre de caractéristiques encodées :",
    len(reference.named_steps["vectorisation"].get_feature_names_out()),
)

display(pd.DataFrame(resultats).round(4))

Durée d’apprentissage et de prédiction : 0.31 secondes
Nombre de caractéristiques encodées : 764


,modele,accuracy,f1_macro,f1_pondere
0,KNN — 3 voisins,0.5534,0.4793,0.5364


## Comparaison avec un SVM linéaire

| Élément | Choix |
| --- | --- |
| Données | Même séparation que KNN |
| Prétraitements | Nouvel encodage et nouvelle mise à l’échelle, ajustés sur l’apprentissage |
| Classificateur | `LinearSVC`, `C=1`, solveur primal |

Cette comparaison porte sur deux modèles simples. Aucun réglage systématique des hyperparamètres n’est effectué.

In [ ]:
modele_svm = Pipeline([
    ("vectorisation", DictVectorizer(sparse=True)),
    ("mise_echelle", StandardScaler(with_mean=False)),
    ("classification", LinearSVC(
        C=1.0,
        dual=False,
        max_iter=10000,
        random_state=RANDOM_STATE,
    )),
])

debut = perf_counter()
modele_svm.fit(X_apprentissage, y_apprentissage)
predictions_svm = modele_svm.predict(X_validation)

nom_modele = "SVM linéaire — C=1"
modeles[nom_modele] = modele_svm
predictions_modeles[nom_modele] = predictions_svm

resultats = [ligne for ligne in resultats if ligne["modele"] != nom_modele]
resultats.append({
    "modele": nom_modele,
    **calculer_scores(y_validation, predictions_svm),
})

print("Durée d’apprentissage et de prédiction :",
      round(perf_counter() - debut, 2), "secondes")

table_resultats = pd.DataFrame(resultats).sort_values(
    "f1_macro", ascending=False
).reset_index(drop=True)

display(table_resultats.round(4))

## Analyse par utilisateur

Le modèle ayant le meilleur F1 macro sur cette validation est utilisé pour l’analyse suivante. Ce critère donne le même poids à chaque utilisateur ; il ne préjuge pas de la métrique officielle Kaggle.

| Mesure | Lecture |
| --- | --- |
| Précision | Fiabilité des prédictions d’un utilisateur |
| Rappel | Part de ses sessions retrouvées |
| F1 | Compromis entre précision et rappel |
| Effectif de validation | Nombre de sessions disponibles pour évaluer cet utilisateur |

Un score fondé sur une ou deux sessions est très sensible à une seule erreur.

In [ ]:
nom_meilleur_modele = table_resultats.iloc[0]["modele"]
modele_retenu = modeles[nom_meilleur_modele]
predictions_validation = predictions_modeles[nom_meilleur_modele]

rapport = classification_report(
    y_validation,
    predictions_validation,
    labels=sorted(y_sessions.unique()),
    output_dict=True,
    zero_division=0,
)

rapport_utilisateurs = pd.DataFrame(rapport).T.loc[
    sorted(y_sessions.unique())
].rename(columns={
    "recall": "rappel",
    "f1-score": "f1",
    "support": "nb_sessions_validation",
})

rapport_utilisateurs["nb_sessions_apprentissage"] = (
    y_apprentissage.value_counts().reindex(rapport_utilisateurs.index)
)

print("Modèle analysé :", nom_meilleur_modele)
print("Les 15 utilisateurs ayant les F1 les plus faibles :")
display(rapport_utilisateurs.sort_values(
    ["f1", "nb_sessions_validation"], ascending=[True, False]
).head(15).round(3))

Modèle analysé : SVM linéaire — C=1
Les 15 utilisateurs ayant les F1 les plus faibles :


,precision,rappel,f1,nb_sessions_validation,nb_sessions_apprentissage
fuy,0.0,0.0,0.0,3.0,12
qqe,0.0,0.0,0.0,3.0,12
xvr,0.0,0.0,0.0,3.0,11
zqs,0.0,0.0,0.0,3.0,13
aho,0.0,0.0,0.0,2.0,8
bos,0.0,0.0,0.0,2.0,7
cje,0.0,0.0,0.0,2.0,6
cvw,0.0,0.0,0.0,2.0,10
fyf,0.0,0.0,0.0,2.0,6
hll,0.0,0.0,0.0,2.0,7


## Analyse des erreurs

Les numéros de session permettent de retrouver les traces d’origine. Ils servent uniquement à l’analyse et ne sont pas utilisés par les modèles.

In [ ]:
analyse_predictions = pd.DataFrame({
    "session_id": df_raw.loc[y_validation.index, "session_id"],
    "util_reel": y_validation,
    "util_predit": predictions_validation,
})
analyse_predictions["correct"] = (
    analyse_predictions["util_reel"] == analyse_predictions["util_predit"]
)
erreurs = analyse_predictions.loc[~analyse_predictions["correct"]].copy()

print("Prédictions correctes :", int(analyse_predictions["correct"].sum()))
print("Erreurs :", len(erreurs))
display(erreurs.head(15))

confusions_frequentes = (
    erreurs.groupby(["util_reel", "util_predit"])
    .size()
    .sort_values(ascending=False)
    .rename("nb_erreurs")
    .reset_index()
)
display(confusions_frequentes.head(10))

Prédictions correctes : 495
Erreurs : 161


,session_id,util_reel,util_predit,correct
1667,1668,ajo,czi,False
1806,1807,opp,zok,False
601,602,slq,guh,False
1914,1915,myl,tzn,False
2993,2994,sbz,itt,False
676,677,zbn,sii,False
2062,2063,snn,iyf,False
2747,2748,zro,nrb,False
2211,2212,mmo,gcr,False
2280,2281,skm,slq,False


,util_reel,util_predit,nb_erreurs
0,skm,slq,3
1,ajo,czi,2
2,xvr,dbi,2
3,qqe,kzt,2
4,hll,jfm,2
5,cjr,wfs,2
6,aho,ufc,1
7,akx,ngs,1
8,akx,szr,1
9,ald,hrj,1


## Contrôles finaux

Vérifier la séparation, les prédictions, la présence du navigateur et la conservation des données brutes.

In [ ]:
assert len(X_apprentissage) + len(X_validation) == len(X_sessions)
assert set(y_apprentissage.index).isdisjoint(y_validation.index)
assert set(y_apprentissage) == set(y_sessions) == set(y_validation)
assert df_raw.equals(df_train[df_raw.columns])
assert all("util" not in ligne and "session_id" not in ligne for ligne in X_sessions)
assert all("navigateur" in ligne for ligne in X_sessions)
assert int(rapport_utilisateurs["nb_sessions_validation"].sum()) == len(y_validation)

for nom, predictions in predictions_modeles.items():
    assert len(predictions) == len(y_validation)
    assert set(predictions).issubset(set(y_apprentissage))
    noms_variables = modeles[nom].named_steps[
        "vectorisation"
    ].get_feature_names_out()
    assert any(nom_variable.startswith("navigateur=") for nom_variable in noms_variables)

assert table_resultats[["accuracy", "f1_macro", "f1_pondere"]].notna().all().all()
print("Tous les contrôles sont validés.")

Tous les contrôles sont validés.


## Synthèse des résultats

Le tableau suivant résume les performances sur la validation et le modèle utilisé pour l’analyse des erreurs.

In [ ]:
synthese = table_resultats.copy()
for colonne in ["accuracy", "f1_macro", "f1_pondere"]:
    synthese[colonne] = (synthese[colonne] * 100).round(2)

synthese = synthese.rename(columns={
    "accuracy": "accuracy (%)",
    "f1_macro": "F1 macro (%)",
    "f1_pondere": "F1 pondéré (%)",
})
display(synthese)
print("Modèle retenu provisoirement pour l’analyse :", nom_meilleur_modele)

,modele,accuracy (%),F1 macro (%),F1 pondéré (%)
0,SVM linéaire — C=1,75.46,69.72,73.88
1,KNN — 3 voisins,55.34,47.93,53.64


Modèle retenu provisoirement pour l’analyse : SVM linéaire — C=1


## Bilan et limites

| Point | Conclusion |
| --- | --- |
| Premier modèle | KNN à trois voisins évalué |
| Comparaison | SVM linéaire évalué sur les mêmes sessions |
| Prétraitements | Ajustés sur l’apprentissage uniquement |
| Analyse | Performances par utilisateur et confusions disponibles |
| Données brutes | Conservées |
| Test Kaggle | Non utilisé |

Les résultats reposent sur une seule séparation aléatoire. Le modèle choisi sur cette validation peut avoir un score optimiste : une validation croisée permettra de confirmer les choix. Les performances par utilisateur sont fragiles lorsque peu de sessions sont disponibles.

Les classes de validation sont déjà connues dans l’apprentissage. Cette évaluation ne mesure ni la reconnaissance d’utilisateurs inconnus ni la détection d’intrusions. Le type exact de moyenne du F1 utilisé par Kaggle reste à confirmer.

### Étapes suivantes

| Étape | Objectif |
| --- | --- |
| Confirmer les choix | Validation croisée stratifiée, au plus quatre plis compte tenu des classes les moins représentées |
| Améliorer | Comparer un petit nombre de paramètres et de groupes de caractéristiques |
| Soumission | Réentraîner la solution retenue sur tout le train et prédire le test dans un notebook distinct |
| Rapport | Documenter les caractéristiques, les métriques, les résultats et leurs limites |